# Data exploration: Tennessee Eastman Process Simulation Dataset


The **Tennessee Eastman Process (TEP) dataset** provides simulated time-series data representative of industrial chemical processes. It serves as a standard benchmark for evaluating fault detection and diagnostic algorithms. By including both nominal and faulty operational scenarios, the dataset allows for evaluating and comparing the performance of monitoring algorithms.

## Dataset overview (from paper's information)

| Parameter | Value |
| :--- | :--- |
| **Dataset** | Tennessee Eastman Process Simulation Dataset |
| **Domain** | Chemical & Process; Manufacturing & Production |
| **Asset / Process** | Chemical Process |
| **Modality** | Time Series |
| **Task** | Anomaly Detection; Fault Diagnosis; Predictive Maintenance; Process Monitoring |
| **Annotation** | Sample Label; Class Label |
| **Source Type** | Simulation |
| **Access** | Zenodo / Dataverse |


## Data Organization and Features

The dataset consists of four DataFrames (*fault_free_training*, *fault_free_testing*, *faulty_testing*, and *faulty_training*) each stored in its own .RData file.

Each dataframe contains 55 columns:

- **Column 1 ('faultNumber')**: Represents the fault type. It ranges from 1 to 20 in the “Faulty” datasets, while the “FaultFree” datasets only contain fault 0 (indicating normal operating conditions).

- **Column 2 ('simulationRun')**: Represents a different random number generator state from which a full TEP dataset was generated, ranging from 1 to 500. Each run was generated using a unique random seed and the seeds for training and testing datasets were non-overlapping.

- **Column 3 ('sample')** : Represents the sequential time step of the simulation. 
- **Columns 4 to 55** contain the **TEP variables**. Those process variables were sampled every 3 minutes. 



## Set up and loading data

In [20]:
import pyreadr
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

current_dir = Path(os.getcwd())
base_data_path = current_dir.parent / "data" / "Tennessee_Eastman_Process_Simulation_Dataset" / "archive"
save_path = current_dir.parent / "data" / "Tennessee_Eastman_Process_Simulation_Dataset" / "processed_csv"
save_path.mkdir(parents=True, exist_ok=True)


Matplotlib is building the font cache; this may take a moment.


### Loading .RData files

In [2]:
files_to_load = {
    "fault_free_test": "TEP_FaultFree_Testing.RData",
    "fault_free_train": "TEP_FaultFree_Training.RData",
    "faulty_test": "TEP_Faulty_Testing.RData",
    "faulty_train": "TEP_Faulty_Training.RData"
}


loaded_data = {}
dfs = {}

for key, filename in files_to_load.items():
    file_path = base_data_path / filename
    
    if file_path.exists():
        r_data = pyreadr.read_r(file_path)
        object_name = list(r_data.keys())[0]
        dfs[key] = r_data[object_name]
        print(f"File {filename} loaded successfully")
    else:
        print(f"Error: File {filename} not found in {base_data_path}")




File TEP_FaultFree_Testing.RData loaded successfully
File TEP_FaultFree_Training.RData loaded successfully
File TEP_Faulty_Testing.RData loaded successfully
File TEP_Faulty_Training.RData loaded successfully


### Saving CSV files

In [5]:
for key, df in dfs.items():
    file_path = save_path / f"{key}.csv"
    
    if file_path.exists():

        print(f"File {file_path.name} already exists. Keeping the original file.")
    else:
        df.to_csv(file_path, index=False)
        print(f"New file {file_path.name} saved successfully.")
    

File fault_free_test.csv already exists. Keeping the original file.
File fault_free_train.csv already exists. Keeping the original file.
File faulty_test.csv already exists. Keeping the original file.
File faulty_train.csv already exists. Keeping the original file.


### Loading CSV files

In [2]:
dfs_csv = {}

for file_path in save_path.glob("*.csv"):
    key = file_path.stem 
    dfs_csv[key] = pd.read_csv(file_path)
    print(f"Loaded: {key}")
print(f"\nLoaded {len(dfs_csv)} CSV files: {list(dfs_csv.keys())}")


df_faulty_test = dfs_csv["faulty_test"]
df_faulty_train = dfs_csv["faulty_train"]
df_fault_free_test = dfs_csv["fault_free_test"]
df_fault_free_train = dfs_csv["fault_free_train"]

Loaded: faulty_test
Loaded: faulty_train
Loaded: fault_free_test
Loaded: fault_free_train

Loaded 4 CSV files: ['faulty_test', 'faulty_train', 'fault_free_test', 'fault_free_train']


## Visualization and analysis

The data is organized hierarchically to represent different industrial operating conditions over time:  
- **Fault Number**: In the "Faulty" datasets, there are 20 distinct fault types.
- **Simulation Run**: For each fault type, the process is simulated 500 times, each using a unique random seed to ensure independent and randomized state conditions.  
- **Sample (Time Steps)**: Each simulation run consists of a sequential series of time-stamped observations:  
    - Training Datasets: Each simulation run contains 500 samples, covering a total duration of 25 hours. Faults were introduced after 1 hour of operation.  
    - Testing Datasets: Each simulation run contains 960 samples, covering a total duration of 48 hours. Faults were introduced after 8 hours of operation.  

### Faulty test dataframe

In [3]:
df_faulty_test.head()

,faultNumber,simulationRun,sample,xmeas_1,xmeas_2,xmeas_3,xmeas_4,xmeas_5,xmeas_6,xmeas_7,...,xmv_2,xmv_3,xmv_4,xmv_5,xmv_6,xmv_7,xmv_8,xmv_9,xmv_10,xmv_11
0,1,1.0,1,0.25171,3672.4,4466.3,9.5122,27.057,42.473,2705.6,...,54.494,24.527,59.710,22.357,40.149,40.074,47.955,47.300,42.100,15.345
1,1,1.0,2,0.25234,3642.2,4568.7,9.4145,26.999,42.586,2705.2,...,53.269,24.465,60.466,22.413,39.956,36.651,45.038,47.502,40.553,16.063
2,1,1.0,3,0.24840,3643.1,4507.5,9.2901,26.927,42.278,2703.5,...,54.000,24.860,60.642,22.199,40.074,41.868,44.553,47.479,41.341,20.452
3,1,1.0,4,0.25153,3628.3,4519.3,9.3347,26.999,42.330,2703.9,...,53.860,24.553,61.908,21.981,40.141,40.066,48.048,47.440,40.780,17.123
4,1,1.0,5,0.21763,3655.8,4571.0,9.3087,26.901,42.402,2707.7,...,53.307,21.775,61.891,22.412,37.696,38.295,44.678,47.530,41.089,18.681


In [4]:
df_faulty_test.shape

(9600000, 55)

In [5]:
print(df_faulty_test.nunique())

faultNumber          20
simulationRun       500
sample              960
xmeas_1          251494
xmeas_2            4612
xmeas_3           13484
xmeas_4           28214
xmeas_5            2757
xmeas_6            4067
xmeas_7            5477
xmeas_8           20174
xmeas_9             137
xmeas_10          64349
xmeas_11          15075
xmeas_12           8131
xmeas_13           5807
xmeas_14           9965
xmeas_15           8383
xmeas_16           5600
xmeas_17           6035
xmeas_18          19534
xmeas_19          98157
xmeas_20          12854
xmeas_21          18579
xmeas_22          17906
xmeas_23          15288
xmeas_24          23844
xmeas_25          16515
xmeas_26          14479
xmeas_27          12491
xmeas_28          17946
xmeas_29          22183
xmeas_30           3338
xmeas_31          24006
xmeas_32          60548
xmeas_33          16911
xmeas_34          14978
xmeas_35          27335
xmeas_36          15471
xmeas_37         187896
xmeas_38          60303
xmeas_39        

### Faulty train dataframe

In [6]:
df_faulty_train.head()

,faultNumber,simulationRun,sample,xmeas_1,xmeas_2,xmeas_3,xmeas_4,xmeas_5,xmeas_6,xmeas_7,...,xmv_2,xmv_3,xmv_4,xmv_5,xmv_6,xmv_7,xmv_8,xmv_9,xmv_10,xmv_11
0,1,1.0,1,0.25038,3674.0,4529.0,9.2320,26.889,42.402,2704.3,...,53.744,24.657,62.544,22.137,39.935,42.323,47.757,47.510,41.258,18.447
1,1,1.0,2,0.25109,3659.4,4556.6,9.4264,26.721,42.576,2705.0,...,53.414,24.588,59.259,22.084,40.176,38.554,43.692,47.427,41.359,17.194
2,1,1.0,3,0.25038,3660.3,4477.8,9.4426,26.875,42.070,2706.2,...,54.357,24.666,61.275,22.380,40.244,38.990,46.699,47.468,41.199,20.530
3,1,1.0,4,0.24977,3661.3,4512.1,9.4776,26.758,42.063,2707.2,...,53.946,24.725,59.856,22.277,40.257,38.072,47.541,47.658,41.643,18.089
4,1,1.0,5,0.29405,3679.0,4497.0,9.3381,26.889,42.650,2705.1,...,53.658,28.797,60.717,21.947,39.144,41.955,47.645,47.346,41.507,18.461


In [7]:
df_faulty_train.shape

(5000000, 55)

In [8]:
print(df_faulty_train.nunique())

faultNumber          20
simulationRun       500
sample              500
xmeas_1          230469
xmeas_2            4534
xmeas_3           13283
xmeas_4           27479
xmeas_5            2703
xmeas_6            3977
xmeas_7            5412
xmeas_8           19815
xmeas_9             138
xmeas_10          62298
xmeas_11          14591
xmeas_12           7924
xmeas_13           5739
xmeas_14           9711
xmeas_15           8141
xmeas_16           5535
xmeas_17           5891
xmeas_18          19176
xmeas_19          80232
xmeas_20          12534
xmeas_21          18365
xmeas_22          17528
xmeas_23          15004
xmeas_24          23214
xmeas_25          16202
xmeas_26          13764
xmeas_27          12210
xmeas_28          15434
xmeas_29          21735
xmeas_30           3249
xmeas_31          23508
xmeas_32          49005
xmeas_33          16541
xmeas_34          14328
xmeas_35          26710
xmeas_36          15089
xmeas_37         149731
xmeas_38          54229
xmeas_39        

### Fault free test dataframe

In [9]:
df_fault_free_test.head()

,faultNumber,simulationRun,sample,xmeas_1,xmeas_2,xmeas_3,xmeas_4,xmeas_5,xmeas_6,xmeas_7,...,xmv_2,xmv_3,xmv_4,xmv_5,xmv_6,xmv_7,xmv_8,xmv_9,xmv_10,xmv_11
0,0,1.0,1,0.25171,3672.4,4466.3,9.5122,27.057,42.473,2705.6,...,54.494,24.527,59.710,22.357,40.149,40.074,47.955,47.300,42.100,15.345
1,0,1.0,2,0.25234,3642.2,4568.7,9.4145,26.999,42.586,2705.2,...,53.269,24.465,60.466,22.413,39.956,36.651,45.038,47.502,40.553,16.063
2,0,1.0,3,0.24840,3643.1,4507.5,9.2901,26.927,42.278,2703.5,...,54.000,24.860,60.642,22.199,40.074,41.868,44.553,47.479,41.341,20.452
3,0,1.0,4,0.25153,3628.3,4519.3,9.3347,26.999,42.330,2703.9,...,53.860,24.553,61.908,21.981,40.141,40.066,48.048,47.440,40.780,17.123
4,0,1.0,5,0.21763,3655.8,4571.0,9.3087,26.901,42.402,2707.7,...,53.307,21.775,61.891,22.412,37.696,38.295,44.678,47.530,41.089,18.681


In [10]:
df_fault_free_test.shape

(207252, 55)

In [11]:
print(df_fault_free_test.nunique())

faultNumber          1
simulationRun      216
sample             960
xmeas_1          17136
xmeas_2           2331
xmeas_3           2689
xmeas_4           5447
xmeas_5           1488
xmeas_6           1538
xmeas_7            569
xmeas_8           3528
xmeas_9             18
xmeas_10          7634
xmeas_11          1691
xmeas_12          6196
xmeas_13           595
xmeas_14          6334
xmeas_15          6272
xmeas_16           513
xmeas_17          3989
xmeas_18          2960
xmeas_19          6723
xmeas_20          1224
xmeas_21          1005
xmeas_22          1852
xmeas_23          1952
xmeas_24          5947
xmeas_25          2052
xmeas_26          6126
xmeas_27          1899
xmeas_28          1676
xmeas_29          2224
xmeas_30           763
xmeas_31          2507
xmeas_32          5947
xmeas_33          2182
xmeas_34          1738
xmeas_35          4029
xmeas_36          3286
xmeas_37         27289
xmeas_38          8671
xmeas_39         16120
xmeas_40          2822
xmeas_41   

### Fault free train dataframe

In [13]:
df_fault_free_train.head()

,faultNumber,simulationRun,sample,xmeas_1,xmeas_2,xmeas_3,xmeas_4,xmeas_5,xmeas_6,xmeas_7,...,xmv_2,xmv_3,xmv_4,xmv_5,xmv_6,xmv_7,xmv_8,xmv_9,xmv_10,xmv_11
0,0.0,1.0,1,0.25038,3674.0,4529.0,9.2320,26.889,42.402,2704.3,...,53.744,24.657,62.544,22.137,39.935,42.323,47.757,47.510,41.258,18.447
1,0.0,1.0,2,0.25109,3659.4,4556.6,9.4264,26.721,42.576,2705.0,...,53.414,24.588,59.259,22.084,40.176,38.554,43.692,47.427,41.359,17.194
2,0.0,1.0,3,0.25038,3660.3,4477.8,9.4426,26.875,42.070,2706.2,...,54.357,24.666,61.275,22.380,40.244,38.990,46.699,47.468,41.199,20.530
3,0.0,1.0,4,0.24977,3661.3,4512.1,9.4776,26.758,42.063,2707.2,...,53.946,24.725,59.856,22.277,40.257,38.072,47.541,47.658,41.643,18.089
4,0.0,1.0,5,0.29405,3679.0,4497.0,9.3381,26.889,42.650,2705.1,...,53.658,28.797,60.717,21.947,39.144,41.955,47.645,47.346,41.507,18.461


In [14]:
df_fault_free_train.shape

(250000, 55)

In [15]:
print(df_fault_free_train.nunique())

faultNumber          1
simulationRun      500
sample             500
xmeas_1          17399
xmeas_2           2384
xmeas_3           2717
xmeas_4           5539
xmeas_5           1500
xmeas_6           1541
xmeas_7            599
xmeas_8           3609
xmeas_9             18
xmeas_10          7782
xmeas_11          1745
xmeas_12          6287
xmeas_13           626
xmeas_14          6428
xmeas_15          6357
xmeas_16           518
xmeas_17          4106
xmeas_18          3021
xmeas_19          6884
xmeas_20          1230
xmeas_21          1016
xmeas_22          1885
xmeas_23          1946
xmeas_24          6070
xmeas_25          2094
xmeas_26          6246
xmeas_27          1961
xmeas_28          1683
xmeas_29          2227
xmeas_30           778
xmeas_31          2545
xmeas_32          6137
xmeas_33          2206
xmeas_34          1764
xmeas_35          4119
xmeas_36          3349
xmeas_37         30719
xmeas_38          8856
xmeas_39         17650
xmeas_40          2894
xmeas_41   

### Fault distribution for faulty dataset

In [17]:
for key, df in dfs_csv.items():
    if 'faulty' in key.lower(): 
        if 'faultNumber' in df.columns:
            counts = df['faultNumber'].value_counts().sort_index()
            print(f"\nFault distribution for '{key}':")
            print(counts)


Fault distribution for 'faulty_test':
faultNumber
1     480000
2     480000
3     480000
4     480000
5     480000
6     480000
7     480000
8     480000
9     480000
10    480000
11    480000
12    480000
13    480000
14    480000
15    480000
16    480000
17    480000
18    480000
19    480000
20    480000
Name: count, dtype: int64

Fault distribution for 'faulty_train':
faultNumber
1     250000
2     250000
3     250000
4     250000
5     250000
6     250000
7     250000
8     250000
9     250000
10    250000
11    250000
12    250000
13    250000
14    250000
15    250000
16    250000
17    250000
18    250000
19    250000
20    250000
Name: count, dtype: int64


### Cheking the presence of null values

In [16]:
for key, df in dfs.items():
    n_nulls = df.isnull().sum().sum()
    print(f"Dataset '{key}' has {n_nulls} missing values.")

Dataset 'fault_free_test' has 0 missing values.
Dataset 'fault_free_train' has 0 missing values.
Dataset 'faulty_test' has 0 missing values.
Dataset 'faulty_train' has 0 missing values.
